In [2]:
import pandas as pd

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train.head()

,SampleID,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,...,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,2276,18,95,Low,High,No,7,96,Medium,Yes,...,Medium,High,Private,Positive,4,Yes,High School,Moderate,Male,70
1,4604,16,89,High,Medium,Yes,8,58,Low,Yes,...,Low,Medium,Public,Neutral,3,No,College,Moderate,Male,67
2,2203,16,69,Medium,High,Yes,6,55,Medium,Yes,...,Medium,High,Public,Negative,1,No,High School,Moderate,Male,62
3,472,11,65,Medium,High,No,7,78,Medium,Yes,...,Low,Medium,Public,Neutral,3,No,Postgraduate,Near,Female,63
4,4061,21,95,Medium,High,Yes,8,57,Low,No,...,Low,Medium,Public,Positive,3,No,High School,Near,Male,69


In [9]:
catCols = train.select_dtypes(include="str").columns
for c in catCols:
    print(f"{c} has {train[c].nunique()} unique columns")
print(train.isna().sum().sum())
# Low cardinality -> onehot

Parental_Involvement has 3 unique columns
Access_to_Resources has 3 unique columns
Extracurricular_Activities has 2 unique columns
Motivation_Level has 3 unique columns
Internet_Access has 2 unique columns
Family_Income has 3 unique columns
Teacher_Quality has 3 unique columns
School_Type has 2 unique columns
Peer_Influence has 3 unique columns
Learning_Disabilities has 2 unique columns
Parental_Education_Level has 3 unique columns
Distance_from_Home has 3 unique columns
Gender has 2 unique columns
0


In [18]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, KFold
from xgboost import XGBRegressor
from scipy.stats import uniform, randint, loguniform

X_train = train.drop(columns=["Exam_Score", "SampleID"])
y_train = train["Exam_Score"]
X_test = test.drop(columns=["SampleID"])

preprocessor = ColumnTransformer(
    transformers=[
        ("oneHot", OneHotEncoder(drop="first", handle_unknown="ignore"), catCols)
    ],
    remainder="passthrough"
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBRegressor(random_state=42))
])

param_dist = {
    "model__n_estimators": randint(100, 1000),
    "model__max_depth": randint(3, 7),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__reg_alpha": [0, 0.01, 0.1, 1],
    "model__min_child_weight": randint(1, 6)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rs = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    random_state=42,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    n_iter=30,
    verbose=2
)

rs.fit(X_train, y_train)
model = rs.best_estimator_

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END model__colsample_bytree=0.749816047538945, model__learning_rate=0.17254716573280354, model__max_depth=5, model__min_child_weight=5, model__n_estimators=120, model__reg_alpha=0.1, model__subsample=0.7783331011414365; total time=   0.1s
[CV] END model__colsample_bytree=0.749816047538945, model__learning_rate=0.17254716573280354, model__max_depth=5, model__min_child_weight=5, model__n_estimators=120, model__reg_alpha=0.1, model__subsample=0.7783331011414365; total time=   0.1s
[CV] END model__colsample_bytree=0.749816047538945, model__learning_rate=0.17254716573280354, model__max_depth=5, model__min_child_weight=5, model__n_estimators=120, model__reg_alpha=0.1, model__subsample=0.7783331011414365; total time=   0.1s
[CV] END model__colsample_bytree=0.749816047538945, model__learning_rate=0.17254716573280354, model__max_depth=5, model__min_child_weight=5, model__n_estimators=120, model__reg_alpha=0.1, model__subsample=0

In [19]:
predictions = model.predict(X_test)
rows = []
for id, pred in zip(test["SampleID"], predictions):
    rows.append({"SampleID": id, "Exam_Score": pred})
sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)